# Demo — Híbrido com reranking: onde a precisão sobe mais por menos esforço

**Objetivo:** ligar o **semantic ranker** do Azure AI Search sobre a mesma busca híbrida do notebook 02 e medir o que muda: a posição do trecho certo, a nota de relevância de cada candidato e o tamanho do contexto. É a linha **Híbrido com reranking** do slide 15, o funil do slide 16 e o degrau 2 do slide 17.

1. **O mesmo híbrido, agora com semantic ranker** → a mesma consulta com dois parâmetros a mais (e um terceiro, opcional, de plano B), e a posição do trecho certo antes e depois;
2. **Antes e depois** → as seis perguntas do slide 14 e a pergunta do reranking: o trecho certo entra no top 5?
3. **O funil do slide 16** → 50 candidatos, nota de 0 a 4, 5 trechos no contexto;
4. **Contexto enxuto e fatura** → os tokens de 20 trechos do híbrido contra os de 5 reordenados, e a resposta com citação.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

**Pré-requisito:** o índice `cap05-acervo`, criado e carregado pelo notebook `tipos-de-rag/00-indice-compartilhado`. Ele já traz a configuração semântica `semantica-padrao`: este notebook não muda nada no índice.

## 0. Preparação

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat, usado na Execução 4)
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`: os **mesmos** valores usados para criar o índice no notebook 00
- `AZURE_SEARCH_ENDPOINT` e `AZURE_SEARCH_API_KEY`

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

**Semantic ranker no serviço:** use o mesmo AI Search do notebook 00, no tier **Basic ou superior** e numa região que ofereça o semantic ranker. Todo serviço começa no plano **gratuito** do semantic ranker, com uma cota mensal de consultas (a página de preços fala nas 1.000 primeiras do mês). Quando ela acaba, a consulta semântica devolve erro de cobrança até o serviço passar para o plano **Standard**, pago por uso, na página **Premium features** do serviço, no portal do Azure. Uma execução completa deste notebook faz 8 consultas semânticas.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "tiktoken>=0.7" "azure-search-documents>=12.0.0,<13"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT") or input(
    "AZURE_SEARCH_ENDPOINT (ex.: https://meu-servico.search.windows.net): "
).strip()
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY") or getpass("AZURE_SEARCH_API_KEY (chave de administração): ")

NOME_INDICE = "cap05-acervo"
credencial_busca = AzureKeyCredential(AZURE_SEARCH_API_KEY)
index_client = SearchIndexClient(AZURE_SEARCH_ENDPOINT, credencial_busca)
search_client = SearchClient(AZURE_SEARCH_ENDPOINT, NOME_INDICE, credencial_busca)

# Este notebook só consulta: quem cria e carrega o índice é o notebook 00.
if NOME_INDICE not in list(index_client.list_index_names()):
    raise RuntimeError("Índice não encontrado. Rode antes o notebook tipos-de-rag/00-indice-compartilhado.")

print(f"Índice '{NOME_INDICE}' encontrado, com {search_client.get_document_count()} trechos.")

### Funções auxiliares

- `gerar_embeddings` e `responder`: as mesmas do notebook 01. A pergunta vira vetor com o mesmo deployment e a mesma dimensão da indexação, e a resposta cita o `id` de cada trecho usado.
- `PERGUNTAS` e `posicao`: o gabarito (`acervo/perguntas.json`) diz qual trecho responde cada pergunta; `posicao` diz em que lugar ele apareceu.
- `CANDIDATOS` e `TRECHOS_NO_CONTEXTO`: os dois números do slide 16, 50 candidatos e 5 trechos no contexto.
- `lugar`, `no_contexto` e `nota` só formatam as tabelas. Quando a nota do reranker não vem, aparece “—”.

In [ ]:
pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores


INSTRUCOES_RAG = """
Responda em português, exclusivamente com base nos trechos fornecidos.
Cite a fonte de cada afirmação no formato [id do trecho].
Se os trechos não trouxerem a informação, responda:
'Não há evidência suficiente nos trechos recuperados.'
""".strip()


def responder(pergunta: str, trechos_recuperados: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_recuperados
    )
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RAG,
        input=f"TRECHOS RECUPERADOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    return resposta.output_text

In [ ]:
import json

PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}

# As seis perguntas do slide 14: três em que o vetor acerta e três em que ele erra.
PERGUNTAS_SLIDE_14 = ["linguagem-natural", "parafrase", "assunto-certo", "codigo-de-norma", "sigla-interna", "string-exata"]


def posicao(ids_retornados: list[str], grupo: list[str]) -> int | None:
    """Posição (1, 2, 3...) do primeiro trecho esperado na lista retornada, ou None se ele não apareceu."""
    for i, id_trecho in enumerate(ids_retornados, start=1):
        if id_trecho in grupo:
            return i
    return None


CANDIDATOS = 50  # slide 16: o reranker reordena no máximo os 50 primeiros resultados
TRECHOS_NO_CONTEXTO = 5  # slide 16: só os melhores viram prompt


def lugar(n: int | None) -> str:
    """Posição como "3º", ou um aviso quando o trecho não veio entre os candidatos."""
    return f"{n}º" if n else f"fora dos {CANDIDATOS}"


def no_contexto(n: int | None) -> str:
    """O trecho fica entre os TRECHOS_NO_CONTEXTO primeiros, os que vão para o modelo?"""
    return "sim" if n and n <= TRECHOS_NO_CONTEXTO else "não"


def nota(valor: float | None) -> float | str:
    """Nota do reranker com duas casas, ou um traço quando a etapa semântica não rodou."""
    return "—" if valor is None else round(valor, 2)

---

# Execução 1 — O mesmo híbrido, agora com semantic ranker

A busca híbrida do notebook 02 junta texto (BM25) e vetor numa só consulta e funde as duas listas por RRF. Aqui ela pede **50 candidatos** (`k = 50` no vetor e `top = 50` na resposta): o semantic ranker reordena no máximo os 50 primeiros, e a Microsoft recomenda entregar os 50 a ele.

`buscar_com_reranking` é a **mesma consulta, com dois parâmetros a mais**:

- `query_type="semantic"` liga o reranker (é o “queryType semantic” do slide 16), um segundo modelo que lê a pergunta e cada trecho juntos e dá a cada um uma nota de relevância;
- `semantic_configuration_name="semantica-padrao"` diz o que ele lê: o título do documento como título, a seção e o conteúdo como texto (configuração criada no notebook 00).

Um terceiro parâmetro, opcional, `semantic_error_mode="partial"`, é o plano B: se o serviço estiver sobrecarregado ou a etapa semântica tiver uma falha passageira, a consulta devolve a ordem do híbrido, sem nota, em vez de erro. As tabelas mostram “—” no lugar da nota.

Cada resultado traz dois números:

- `@search.score`, o **score RRF** do híbrido: cada lista dá 1/60 ao 1º, 1/61 ao 2º, e assim por diante (é o `1/(60 + rank)` da documentação, com o rank contado a partir de zero), e o trecho soma o que recebeu das duas. Quem fica em 1º nas duas listas chega a 2/60, cerca de 0,033. Serve só para ordenar.
- `@search.reranker_score`, a **nota do reranker**, de 0 a 4 (4 = altamente relevante, responde por completo; 0 = irrelevante). Com reranking, a ordem é a da nota.

A pergunta é a do gabarito para este degrau: o marido que vai junto na viagem. O trecho que responde é “Despesas que não são reembolsadas”, que fala em **acompanhantes** e **cônjuge**, não em “marido”.

### O que esperar e por que pode variar

- No híbrido, tendem a subir os trechos com mais palavras em comum com a pergunta (“reembolso”, “passagem”, “viagem a trabalho”), como “Como pedir o reembolso de uma viagem” e “Passagens aéreas e deslocamento”. O trecho certo tende a vir abaixo deles.
- Com reranking, ele tende a subir para o topo, com a maior nota. Se já vier no topo do híbrido, o reranker não tem o que corrigir: compare então as notas, que medem a distância entre o trecho que responde e os que só se parecem com a pergunta.
- As notas mudam um pouco entre execuções e com atualizações do modelo de ranking (a própria Microsoft avisa): leia posições e faixas de nota, não casas decimais.

In [ ]:
from azure.search.documents.models import VectorizedQuery


def buscar_hibrida(pergunta: str, top: int = CANDIDATOS, **parametros_extras) -> list[dict]:
    """Busca híbrida: texto (BM25) e vetor na mesma consulta, com as duas listas fundidas por RRF."""
    vetor = gerar_embeddings([pergunta])[0]
    resultados = search_client.search(
        search_text=pergunta,
        vector_queries=[VectorizedQuery(vector=vetor, k_nearest_neighbors=CANDIDATOS, fields="embedding")],
        select=["id", "titulo", "secao", "conteudo", "fonte"],
        top=top,
        **parametros_extras,
    )
    return [
        {
            "id": r["id"],
            "titulo": r["titulo"],
            "secao": r["secao"],
            "conteudo": r["conteudo"],
            "fonte": r["fonte"],
            "score": r["@search.score"],
            "nota_reranker": r["@search.reranker_score"],  # None quando o reranker não rodou
        }
        for r in resultados
    ]


def buscar_com_reranking(pergunta: str, top: int = CANDIDATOS) -> list[dict]:
    """A mesma busca híbrida, com o semantic ranker reordenando os candidatos."""
    return buscar_hibrida(
        pergunta,
        top=top,
        query_type="semantic",  # liga o reranker
        semantic_configuration_name="semantica-padrao",  # o que ele lê (notebook 00)
        semantic_error_mode="partial",  # se ele falhar, volta a ordem do híbrido, sem nota
    )


pergunta_reranking = PERGUNTAS["reranking"]["pergunta"]
trecho_certo = PERGUNTAS["reranking"]["trechos_esperados"][0]

hibrido = buscar_hibrida(pergunta_reranking)
com_reranking = buscar_com_reranking(pergunta_reranking)

antes = posicao([t["id"] for t in hibrido], trecho_certo)
depois = posicao([t["id"] for t in com_reranking], trecho_certo)
lugar_no_hibrido = {t["id"]: i for i, t in enumerate(hibrido, start=1)}

display(Markdown(
    f"**Pergunta:** {pergunta_reranking}  \n"
    f"**Trecho certo:** `{trecho_certo[0]}`  \n"
    f"**Posição do trecho certo:** {lugar(antes)} no híbrido → {lugar(depois)} com reranking"
))
display(pd.DataFrame([
    {
        "posição": i,
        "nota do reranker (0 a 4)": nota(t["nota_reranker"]),
        "score RRF": round(t["score"], 4),
        "posição no híbrido": lugar(lugar_no_hibrido.get(t["id"])),
        "trecho": f"{t['titulo']} — {t['secao']}",
        "é o trecho certo?": "sim" if t["id"] in trecho_certo else "",
    }
    for i, t in enumerate(com_reranking[:TRECHOS_NO_CONTEXTO], start=1)
]))

if all(t["nota_reranker"] is None for t in com_reranking):
    print("Atenção: nenhum trecho veio com nota. A etapa semântica não rodou, e a ordem acima é a do híbrido.")

### O que observar

- A coluna **posição no híbrido** mostra de onde veio cada um dos 5 primeiros: o reranker não busca nada novo, só reordena os mesmos candidatos.
- Compare a **nota** do trecho certo com a dos vizinhos. O score RRF dos cinco tende a ficar na mesma faixa, porque todos se parecem com a pergunta; a nota separa o trecho que responde dos que só falam de reembolso.

### Leitura arquitetural: configuração, não reengenharia (slide 16)

- `buscar_com_reranking` é `buscar_hibrida` com dois parâmetros a mais (e um terceiro, opcional, de plano B: `semantic_error_mode`). Nada mudou no chunking, no embedding ou no índice.
- O preço é **uma chamada a mais** (slide 15): o segundo modelo roda dentro do AI Search, na mesma requisição, soma latência à consulta e, depois da cota gratuita, é cobrado por consulta.
- O ganho pode aparecer duas vezes: o trecho certo tende a subir e, com ele no topo, o contexto pode encolher (Execução 4).

---

# Execução 2 — Antes e depois

As seis perguntas do slide 14 e a pergunta do reranking, cada uma nas duas versões da busca. A coluna que decide é **no top 5: antes → depois**, porque os 5 primeiros são o que vai para o contexto do modelo. É o sintoma do degrau 2 do slide 17: o assunto certo vem, mas o trecho que responde fica fora do contexto.

### O que esperar e por que pode variar

- Onde o híbrido já acerta (linguagem natural, código de norma, string exata), o reranker tende a manter o trecho certo no topo: há pouco a melhorar.
- O ganho tende a aparecer quando as palavras da pergunta não são as do trecho: paráfrase, sinônimo e a pergunta do reranking.
- O reranker lê texto, não confere identificador. Entre irmãos quase idênticos (SKU 48213-B x C x D, ATLAS x ÓRION x HÉLIOS), ele pode trocar a ordem. Se acontecer, é a lição: quem garante a string exata é o BM25 do degrau 1.

In [ ]:
linhas = []
for pid in PERGUNTAS_SLIDE_14 + ["reranking"]:
    p = PERGUNTAS[pid]
    grupo = p["trechos_esperados"][0]
    reordenados = buscar_com_reranking(p["pergunta"])
    antes = posicao([t["id"] for t in buscar_hibrida(p["pergunta"])], grupo)
    depois = posicao([t["id"] for t in reordenados], grupo)
    linhas.append({
        "pergunta": p["pergunta"],
        "categoria": p["categoria"],
        "posição no híbrido": lugar(antes),
        "posição com reranking": lugar(depois),
        "nota do trecho certo": nota(reordenados[depois - 1]["nota_reranker"]) if depois else "—",
        "no top 5: antes → depois": f"{no_contexto(antes)} → {no_contexto(depois)}",
    })

display(pd.DataFrame(linhas))

### O que observar

- A coluna **categoria** é a regra geral do slide; as posições são o que aconteceu neste acervo, com este modelo.
- Linhas com **não → sim** são o caso do degrau 2 do slide 17: o trecho certo vinha fora do contexto, e o reranker o trouxe para dentro.
- **sim → sim**, na mesma posição ou numa melhor: o reranker não atrapalhou o que o híbrido já acertava.
- Se todas as linhas vierem **sim → sim**, neste acervo pequeno o híbrido já pôs o trecho certo entre os 5. Olhe então a mudança de posição e a nota do trecho certo: é a nota que dá segurança para encolher o contexto na Execução 4.
- Uma linha **sim → não**, se aparecer, merece anotação: é o tipo de regressão que só a medição com perguntas de resposta conhecida revela (slide 27).

### Leitura arquitetural

- O reranker não busca de novo no acervo: ele reordena os 50 candidatos que a busca trouxe. Se o trecho certo não estiver entre eles, nenhum reranker o encontra.
- Por isso o degrau 2 vem depois do degrau 1 (slide 17): o híbrido põe o trecho certo entre os candidatos; o reranker o põe entre os 5 que viram contexto.

---

# Execução 3 — O funil do slide 16

O slide 16 resume o reranking em três números: **50** candidatos, nota de **0 a 4**, **5** trechos no contexto. Voltamos à pergunta do reranking (o resultado da Execução 1, sem nova consulta) e contamos quantos dos 50 candidatos ficaram em cada faixa de nota.

Neste acervo de 50 trechos, os 50 candidatos são o acervo inteiro. Num acervo real, seriam os 50 melhores entre milhares.

Como a Microsoft descreve as notas: **4**, o trecho é altamente relevante e responde por completo; **3**, é relevante, mas falta detalhe; **2**, responde em parte; **1**, é relacionado e responde uma pequena parte; **0**, é irrelevante.

In [ ]:
notas = [t["nota_reranker"] for t in com_reranking]
com_nota = [n for n in notas if n is not None]
faixas = [
    ("3 a 4", "relevante; com 4, responde por completo", sum(n >= 3 for n in com_nota)),
    ("2 a 3", "responde em parte", sum(2 <= n < 3 for n in com_nota)),
    ("abaixo de 2", "pouco ou nada relevante para a pergunta", sum(n < 2 for n in com_nota)),
    ("sem nota", "a etapa semântica não rodou", len(notas) - len(com_nota)),
]

display(Markdown(f"**Pergunta:** {pergunta_reranking}"))
display(pd.DataFrame([
    {"nota do reranker": faixa, "o que significa": leitura, "candidatos": quantos}
    for faixa, leitura, quantos in faixas
]))

# O corte é por posição: entram os primeiros da lista já reordenada pela nota.
notas_altas_no_contexto = sum(n >= 3 for n in com_nota[:TRECHOS_NO_CONTEXTO])
detalhe = f"{notas_altas_no_contexto} deles com nota 3 ou mais" if com_nota else "sem nota: a etapa semântica não rodou"
print(
    f"Funil do slide 16: {len(com_reranking)} candidatos → nota de 0 a 4 → "
    f"os {TRECHOS_NO_CONTEXTO} primeiros vão para o contexto ({detalhe})."
)
print(
    f"Nota do {TRECHOS_NO_CONTEXTO}º colocado, o último que entra: {nota(notas[TRECHOS_NO_CONTEXTO - 1])}. "
    f"Nota do {TRECHOS_NO_CONTEXTO + 1}º, o primeiro que fica de fora: {nota(notas[TRECHOS_NO_CONTEXTO])}."
)

### O que observar

- Em geral, poucos candidatos passam de 3, e a maioria fica abaixo de 2. Neste acervo pequeno, os 50 candidatos são o acervo inteiro; num acervo grande, eles viriam porque se **parecem** com a pergunta, não porque a **respondem**: a busca híbrida devolve por similaridade, proximidade, não utilidade (slide 16).
- O corte em 5 é decisão de arquitetura, não do reranker (slide 9: 3 a 5 trechos, com teto de tokens). Compare a nota do 5º com a do 6º: se a do 6º ainda for alta, vale discutir o corte; se for baixa, mandar mais trechos só paga token.
- A distribuição das notas varia um pouco entre execuções. Se for usar um limiar de nota (por exemplo, “só entra nota 2 ou mais”), não o faça granular demais.

---

# Execução 4 — Contexto enxuto e fatura

O slide 16 diz que o reranking “ainda reduz a fatura”. A conta é de tokens de entrada: cada trecho recuperado é token de entrada, em toda chamada, para sempre (slide 7). Para a mesma pergunta, comparamos dois contextos, montados no formato que `responder` usa:

- **20 primeiros do híbrido:** o “mandar vinte trechos ao modelo” do anti-padrão 6 do slide 28;
- **5 primeiros com reranking:** o contexto enxuto.

Os tokens são contados com o tokenizador `o200k_base`, dos modelos GPT mais recentes (a mesma régua do notebook 00). Depois, a resposta com citação a partir dos 5 trechos.

### O que esperar e por que pode variar

- O contexto de 5 trechos tende a ficar em torno de um quarto dos tokens do de 20, porque os trechos deste acervo têm tamanhos parecidos.
- O que decide é a coluna **o trecho certo está?**: o contexto enxuto só é melhor se continuar contendo o trecho que responde.

In [ ]:
import tiktoken

codificador = tiktoken.get_encoding("o200k_base")
PERGUNTAS_POR_DIA = 50_000  # a mesma escala do Cap. 1


def tokens_de_contexto(trechos_enviados: list[dict]) -> int:
    """Tokens do bloco de trechos, montado no mesmo formato que responder() usa."""
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_enviados
    )
    return len(codificador.encode(contexto))


contextos = {
    "20 primeiros do híbrido": hibrido[:20],
    f"{TRECHOS_NO_CONTEXTO} primeiros com reranking": com_reranking[:TRECHOS_NO_CONTEXTO],
}
linhas = []
for nome, enviados in contextos.items():
    lugar_certo = posicao([t["id"] for t in enviados], trecho_certo)
    tokens = tokens_de_contexto(enviados)
    linhas.append({
        "contexto": nome,
        "tokens de contexto": tokens,
        "o trecho certo está?": f"sim, em {lugar_certo}º" if lugar_certo else "não",
        "milhões de tokens por dia (50 mil perguntas)": round(tokens * PERGUNTAS_POR_DIA / 1_000_000, 1),
    })

display(Markdown(f"**Pergunta:** {pergunta_reranking}"))
display(pd.DataFrame(linhas))
print(
    f"O contexto com reranking usa {linhas[1]['tokens de contexto'] / linhas[0]['tokens de contexto']:.0%} "
    "dos tokens do contexto com 20 trechos."
)

In [ ]:
contexto_enxuto = com_reranking[:TRECHOS_NO_CONTEXTO]
lugar_certo = posicao([t["id"] for t in contexto_enxuto], trecho_certo)
enviados = ", ".join(f"`{t['id']}`" for t in contexto_enxuto)
situacao = f"sim, em {lugar_certo}º lugar" if lugar_certo else "não"

display(Markdown(f"### {pergunta_reranking}"))
display(Markdown(f"**Trechos enviados ao modelo:** {enviados}  \n**O trecho certo está entre eles?** {situacao}"))
display(Markdown(responder(pergunta_reranking, contexto_enxuto)))

### O que observar

- Se o trecho “Despesas que não são reembolsadas” estiver entre os 5, a resposta tende a dizer que a passagem do cônjuge não é reembolsada, citando `[financeiro_reembolso-de-despesas-de-viagem_04]`. Se não estiver, o modelo diz que não há evidência suficiente: o erro nasceu na recuperação (slide 29).
- Olhe a última coluna da tabela: a diferença por pergunta parece pequena; na escala de 50 mil perguntas por dia, vira milhões de tokens.

### Leitura arquitetural

- **Tokens de contexto por pergunta** é a métrica que a fatura enxerga (slide 27). O reranking a reduz sem trocar o modelo de geração nem reindexar nada.
- O reranker também tem preço: depois da cota gratuita, cada consulta semântica é cobrada. A conta é uma consulta semântica a mais contra tokens de entrada a menos em toda chamada ao modelo.
- Montar o contexto é a decisão mais barata de mudar (slide 10: “é só código”): aqui, mudar o número de trechos é mudar uma constante.

### Fala sugerida

> “Reranking melhora a resposta sem tocar em chunking, embedding ou índice, e ainda reduz token. É configuração, não reengenharia.”

---

# Quadro final — o que cada execução provou?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | A mesma consulta híbrida, com dois parâmetros a mais (e um terceiro, opcional, de plano B), reordena os 50 candidatos por uma nota de 0 a 4 | Ligar o semantic ranker é configuração da consulta, não reengenharia |
| 2 | O ganho tende a aparecer onde as palavras da pergunta não são as do trecho; o reranker só reordena o que a busca trouxe | Subir para reranking quando o trecho certo vem, mas fica fora do top 5 (slide 17, degrau 2) |
| 3 | Dos 50 candidatos, poucos tendem a ter nota alta: similaridade não é utilidade | Ordenar pela nota e cortar o contexto em 3 a 5 trechos |
| 4 | Cinco trechos reordenados custam uma fração dos tokens de vinte | Medir tokens de contexto por pergunta, a métrica que a fatura enxerga |

## Frase de fechamento

> **Cinco trechos bons custam menos que vinte medianos e respondem melhor. É a regra de contexto enxuto do Capítulo 4, aplicada aqui.**

---

# Referências oficiais

- Azure AI Search — visão geral do semantic ranker (como a nota de 0 a 4 é dada):  
  https://learn.microsoft.com/azure/search/semantic-search-overview

- Azure AI Search — adicionar o semantic ranker a uma consulta:  
  https://learn.microsoft.com/azure/search/semantic-how-to-query-request

- Azure AI Search — consulta híbrida com semantic ranker (k = 50):  
  https://learn.microsoft.com/azure/search/hybrid-search-how-to-query

- Azure AI Search — planos de cobrança do semantic ranker (gratuito e Standard):  
  https://learn.microsoft.com/azure/search/semantic-how-to-enable-disable